# Coupled RAFT and DEQ-Flow in SILVA

The equilibrium state is $(h,u)$:

$$
h^+=\operatorname{ConvGRU}(h,c,m(u,C(u))),\qquad
u^+=u+\Delta_\theta(h^+).
$$

This package-native case exposes residual-encoder stages and stride,
correlation pyramid levels and radius, motion/GRU widths, global
aggregation, solver and gradient rules, sparse correction indices,
learned convex upsampling, fixed-point reuse, and custom encoder or
update modules.

<!-- silva-numbered-citations:start -->
**Numbered literature:** [[22]](https://jseluis.github.io/silva-networks/paper/references/#ref-22), [[23]](https://jseluis.github.io/silva-networks/paper/references/#ref-23), [[24]](https://jseluis.github.io/silva-networks/paper/references/#ref-24). Each number opens the complete citation and its primary external source.
<!-- silva-numbered-citations:end -->


In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [Path.cwd(), Path("/content/silva-networks"), Path("/content/drive/MyDrive/silva-networks")]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [2]:
import torch

from silva_networks import (
    SILVARAFTDEQ,
    SolverConfig,
    make_silva_translation_flow_batch,
    silva_flow_fixed_point_correction_loss,
)

torch.manual_seed(13)
batch = make_silva_translation_flow_batch(
    batch_size=1, channels=1, height=8, width=8, shift=(1.0, 0.0)
)
config = SolverConfig(
    solver="picard",
    max_iter=3,
    alpha=0.5,
    indexing=(1, 2),
    backward_mode="implicit",
    backward_solver="gmres",
    backward_max_iter=8,
)
model = SILVARAFTDEQ(
    in_channels=1,
    feature_dim=8,
    hidden_dim=4,
    context_dim=4,
    encoder_channels=(4,),
    encoder_residual_blocks=1,
    encoder_dropout=0.0,
    output_stride=2,
    corr_levels=2,
    corr_radius=1,
    motion_dim=8,
    flow_head_dim=8,
    gru_kernel_size=3,
    correlation_hidden_dims=(8, 8),
    flow_hidden_dims=(8, 4),
    correction_steps=1,
    config=config,
)

## Solve, Sparse Corrections, and Exact Gradient

`indexing` stores selected numerical states. Short differentiable
corrections turn them into auxiliary flow predictions without
retaining the entire forward solver graph.

In [3]:
result = model(batch.image1, batch.image2, return_result=True)
predictions = result.flow_sequence or [result.flow]
loss = silva_flow_fixed_point_correction_loss(
    predictions, batch.flow, valid=batch.valid, gamma=0.8
)
loss.backward()
finite_gradients = all(
    parameter.grad is None or torch.isfinite(parameter.grad).all()
    for parameter in model.parameters()
)
print("flow", result.flow.shape)
print("low-resolution state", result.low_resolution_flow.shape)
print("correction predictions", len(predictions))
print("finite gradients", bool(finite_gradients))

flow torch.Size([1, 2, 8, 8])
low-resolution state torch.Size([1, 2, 4, 4])
correction predictions 3
finite gradients True


## Reuse the Fixed Point

A previous hidden/flow equilibrium can initialize a related image
pair. Whether this is appropriate across frames or augmentations is
an experiment choice.

In [4]:
reused = model(batch.image1, batch.image2, cached_state=result.cached_state)
print("reused flow", reused.shape)

reused flow torch.Size([1, 2, 8, 8])


## Reproduction Boundary and Citations

This compact run validates the coupled state, correlation/GRU update,
learned upsampling, correction loss, implicit backward path, and
reuse contract. Paper metrics require the source dataset mixtures,
augmentations, schedules, evaluation code, resolution, and model
dimensions.

Cite RAFT for all-pairs correlation and recurrent refinement,
DEQ-Flow for the equilibrium optical-flow formulation and sparse
correction/reuse strategy, and SILVA Networks for this generalized
package API:
https://github.com/jseluis/silva-networks
https://doi.org/10.5281/zenodo.21770098

## From 13 Raft Deq Flow to a Custom SILVA Family

The construction in this notebook can be separated into the universal
conditioned-equilibrium contract

$$
z_0=I_\eta(x),\qquad
z^\star=T_\theta(z^\star,x),\qquad
\widehat y=Q_\psi(z^\star).
$$

For this topic:

| Part | Concrete interpretation |
| --- | --- |
| Equilibrium state | the flow field, optionally coupled to a recurrent hidden state |
| Condition | image features, correlation volumes, context, and initial flow |
| Repeated computation | the tied correlation-conditioned refinement update |
| Required invariants | flow shape, coordinate convention, image resolution, and warping domain |
| Replaceable components | feature/context encoders, correlation, update block, transition, upsampler, and solver |

The initializer and source path are evaluated outside or alongside the root
solve. Only the state-preserving transition is repeated. Replacing an internal
architecture does not change this equation, provided the transition still maps
the same state space into itself.


### Replace the Flow Transition

The complete refinement can be supplied as
`transition_module(flow, fmap1, fmap2, correlation)`. Feature and context
encoders and the update block are independently replaceable.

```python
flow_model = SILVAOpticalFlowDEQ(
    feature_dim=feature_dim,
    encoder_module=my_feature_encoder,
    update_block=my_update_block,
    transition_module=my_flow_transition,
    config=solver_config,
)
```


In [5]:
import torch as silva_extension_torch
from torch import nn as silva_extension_nn

from silva_networks import (
    SILVAConditionedEquilibrium,
    SILVAZeroInitializer,
    SolverConfig,
    validate_silva_transition,
)


class NotebookExtensionTransition(silva_extension_nn.Module):
    def __init__(self, condition_dim=2, state_dim=3):
        super().__init__()
        self.source = silva_extension_nn.Linear(condition_dim, state_dim)
        self.state_field = silva_extension_nn.Sequential(
            silva_extension_nn.Linear(state_dim, 2 * state_dim),
            silva_extension_nn.Tanh(),
            silva_extension_nn.Linear(2 * state_dim, state_dim),
        )

    def forward(self, state, condition):
        return silva_extension_torch.tanh(
            self.source(condition) + 0.15 * self.state_field(state)
        )


silva_extension_torch.manual_seed(610)
notebook_condition = silva_extension_torch.linspace(-1.0, 1.0, 8).reshape(4, 2)
notebook_state0 = silva_extension_torch.zeros(4, 3)
notebook_transition = NotebookExtensionTransition()

notebook_report = validate_silva_transition(
    notebook_transition,
    notebook_state0,
    notebook_condition,
)
assert notebook_report.valid

with silva_extension_torch.no_grad():
    notebook_reference_step = silva_extension_torch.tanh(
        notebook_transition.source(notebook_condition)
        + 0.15 * notebook_transition.state_field(notebook_state0)
    )
silva_extension_torch.testing.assert_close(
    notebook_transition(notebook_state0, notebook_condition),
    notebook_reference_step,
)

notebook_custom_model = SILVAConditionedEquilibrium(
    notebook_transition,
    SILVAZeroInitializer(3),
    readout=silva_extension_nn.Linear(3, 1),
    config=SolverConfig(
        solver="picard",
        max_iter=40,
        tol=1e-7,
        backward_mode="implicit",
        backward_solver="gmres",
        anderson_batch_dims=1,
    ),
)
notebook_custom_result = notebook_custom_model(
    notebook_condition,
    return_result=True,
)
assert notebook_custom_result.output.shape == (4, 1)
assert notebook_custom_result.solver_result.residual < 1e-5

notebook_custom_result.output.square().mean().backward()
assert all(
    parameter.grad is not None and silva_extension_torch.isfinite(parameter.grad).all()
    for parameter in notebook_custom_model.parameters()
)
print("custom transition:", notebook_report)
print("equilibrium residual:", notebook_custom_result.solver_result.residual)


custom transition: SILVATransitionReport(state_shape=(4, 3), output_shape=(4, 3), preserves_shape=True, preserves_device=True, preserves_dtype=True, finite=True, differentiable=True, parameter_count=54)
equilibrium residual: 5.960464477539063e-08


## Numerical Equivalence, Compact Reproduction, and Scale

Before training, compare one packaged transition with an independently written
update:

$$
e_{\mathrm{step}}
=\frac{\|T_\theta(z,x)-T_{\mathrm{ref}}(z,x)\|_2}
{\|T_{\mathrm{ref}}(z,x)\|_2+\varepsilon}.
$$

After solving, report the fixed-point residual separately:

$$
e_{\mathrm{fp}}
=\frac{\|T_\theta(z^\star,x)-z^\star\|_2}
{\|z^\star\|_2+\varepsilon}.
$$

For this notebook, a compact reproduction must declare and assert
**endpoint error, warp error, correction loss, and fixed-point residual**. A full experiment must additionally record the
source dataset version and split, preprocessing, architecture widths, solver
and optimizer schedules, random seeds, baseline configuration, checkpoints,
and every deviation from the cited protocol.

The principal scaling axes are **image resolution, correlation radius/levels, hidden width, and solver budget**. Increase one axis at
a time, retain the compact deterministic case as a regression test, and record
task error, domain-specific residual, forward residual, backward linear
residual, memory use, and runtime independently.

### Extension Exercises

1. Replace one component from this notebook while preserving its state and
   domain invariants.
2. Write the replacement first as an independent reference function, then as
   a module, and assert one-step equivalence.
3. Compare two solver configurations on the identical trained transition.
4. Add a compact baseline and a predeclared metric threshold.
5. Create a full-scale configuration without weakening the compact tests.

The complete authoring protocol is documented in
[Extending SILVA](https://jseluis.github.io/silva-networks/learn/extending-silva/).


In [6]:
notebook_reproduction_record = {
    "notebook": '13_raft_deq_flow.ipynb',
    "state": 'the flow field, optionally coupled to a recurrent hidden state',
    "condition": 'image features, correlation volumes, context, and initial flow',
    "transition": 'the tied correlation-conditioned refinement update',
    "invariants": 'flow shape, coordinate convention, image resolution, and warping domain',
    "compact_metric": 'endpoint error, warp error, correction loss, and fixed-point residual',
    "scale_axis": 'image resolution, correlation radius/levels, hidden width, and solver budget',
}
assert all(notebook_reproduction_record.values())
notebook_reproduction_record


{'notebook': '13_raft_deq_flow.ipynb',
 'state': 'the flow field, optionally coupled to a recurrent hidden state',
 'condition': 'image features, correlation volumes, context, and initial flow',
 'transition': 'the tied correlation-conditioned refinement update',
 'invariants': 'flow shape, coordinate convention, image resolution, and warping domain',
 'compact_metric': 'endpoint error, warp error, correction loss, and fixed-point residual',
 'scale_axis': 'image resolution, correlation radius/levels, hidden width, and solver budget'}

## Where to Go Next

| Question | Page |
| --- | --- |
| How is the coupled flow fixed point derived? | [DEQ Engine and Optical Flow](https://jseluis.github.io/silva-networks/learn/deq-engine-and-flow/) |
| Where is the same flow case available as a script? | [RAFT and DEQ-Flow Example](https://jseluis.github.io/silva-networks/examples/raft-deq-flow/) |
| Which flow controls and results are public? | [Optical Flow API](https://jseluis.github.io/silva-networks/api/flow/) |
